In [1]:
import pandas as pd

In [2]:
female = pd.read_csv('/Users/derekebowman/Downloads/24W_females_Integrated_meanRank.tsv', sep='\t')
male = pd.read_csv('/Users/derekebowman/Downloads/24W_males_Integrated_meanRank.tsv', sep='\t')

In [3]:
import pandas as pd
import numpy as np

def top_shared_tfs(male: pd.DataFrame, female: pd.DataFrame, n: int = 10, how: str = "mean") -> pd.DataFrame:
    """
    Return top-N TFs present in both dataframes by lowest combined rank.

    Parameters
    ----------
    male, female : DataFrames with columns at least ['TF','Rank'].
    n : number of TFs to return.
    how : 'mean' (default), 'min', or 'max' to combine the two ranks.

    Returns
    -------
    DataFrame with columns: TF, rank_male, rank_female, rank_mean, rank_min, rank_max
    sorted by the chosen combination metric.
    """
    # ensure numeric ranks
    for df in (male, female):
        df["Rank"] = pd.to_numeric(df["Rank"], errors="coerce")

    # if duplicates per TF exist, keep the best (lowest) rank per TF within each DF
    m_best = male.dropna(subset=["TF", "Rank"]).groupby("TF", as_index=False)["Rank"].min().rename(columns={"Rank": "rank_male"})
    f_best = female.dropna(subset=["TF", "Rank"]).groupby("TF", as_index=False)["Rank"].min().rename(columns={"Rank": "rank_female"})

    # intersection
    both = m_best.merge(f_best, on="TF", how="inner")
    if both.empty:
        return both  # nothing in common

    # combined metrics
    both["rank_mean"] = both[["rank_male", "rank_female"]].mean(axis=1)
    both["rank_min"]  = both[["rank_male", "rank_female"]].min(axis=1)
    both["rank_max"]  = both[["rank_male", "rank_female"]].max(axis=1)

    key = {"mean": "rank_mean", "min": "rank_min", "max": "rank_max"}.get(how.lower())
    if key is None:
        raise ValueError("how must be one of {'mean','min','max'}")

    out = both.sort_values(key, ascending=True, kind="mergesort").head(n).reset_index(drop=True)
    return out



In [4]:

# --- Example usage ---
result = top_shared_tfs(male, female, n=10, how="mean")

print(result)
# If you just want the TF names:
print(result["TF"].tolist())
genes_of_interest = result["TF"].tolist()

       TF  rank_male  rank_female  rank_mean  rank_min  rank_max
0   SNAI2         21            6       13.5         6        21
1   TSHZ3         14           32       23.0        14        32
2   NKX21         37           10       23.5        10        37
3  TWIST1         30           19       24.5        19        30
4  TWIST2          4           49       26.5         4        49
5   FOXL1          9           45       27.0         9        45
6   PPARG         44           17       30.5        17        44
7   FOXF2         50           12       31.0        12        50
8     MKX         15           47       31.0        15        47
9   SOX11         53           13       33.0        13        53
['SNAI2', 'TSHZ3', 'NKX21', 'TWIST1', 'TWIST2', 'FOXL1', 'PPARG', 'FOXF2', 'MKX', 'SOX11']


In [5]:
genes_of_interest

['SNAI2',
 'TSHZ3',
 'NKX21',
 'TWIST1',
 'TWIST2',
 'FOXL1',
 'PPARG',
 'FOXF2',
 'MKX',
 'SOX11']

### These TFs are now used in CellOracle's workflow. 